# 3. Metodología

El presente estudio corresponde a un **análisis exploratorio de datos (EDA) de corte longitudinal**, basado en registros administrativos de mortalidad en Estados Unidos durante el período 1999–2017. El enfoque metodológico combina estadística descriptiva, análisis de series de tiempo, clustering y modelado predictivo.

## 3.1. Fuente de datos y preprocesamiento

El conjunto contiene registros anuales de mortalidad por causa y estado, incluyendo la tasa ajustada por edad (*age-adjusted death rate*), que estandariza las tasas eliminando el efecto de diferencias en la estructura etaria entre poblaciones, permitiendo comparaciones entre estados y años.

El preprocesamiento incluyó estandarización de nombres de variables, corrección de formatos numéricos inconsistentes, conversión de tipos de datos y filtrado del período 1999–2017.

## 3.2. Estadística descriptiva

Como punto de partida se construyeron diagramas de barras comparativos para analizar la distribución porcentual de las causas de muerte de 1999–2017 y gráficos de líneas para identificar la evolución temporal de las tasas ajustadas por causa a nivel nacional.

## 3.3. Análisis de series de tiempo

- **Estacionariedad:** Evaluada mediante la prueba de Dickey-Fuller Aumentada (ADF), cuya hipótesis nula establece que la serie tiene raíz unitaria (no estacionaria). Un valor p < 0.05 indica estacionariedad.
- **ACF y PACF:** Se analizaron las funciones de autocorrelación (ACF) y autocorrelación parcial (PACF) para identificar patrones de dependencia temporal y orientar la selección de parámetros del modelo.

## 3.4. Análisis geográfico

Se calcularon tasas ajustadas promedio por estado para identificar disparidades territoriales. Se construyeron mapas coropléticos interactivos y visualizaciones comparativas de los estados con mayor y menor carga de mortalidad.

## 3.5. Análisis de clustering – K-Means

Para identificar grupos de estados con perfiles de mortalidad similares se aplicó K-Means. Los pasos fueron:

1. **Estandarización** de variables (media 0, desviación estándar 1).
2. **Método del codo** para determinar el número óptimo de clústeres (k=4).
3. **Visualización PCA** para representar en dos dimensiones la separación entre clústeres.

## 3.6. Modelo predictivo – ARIMA

Para proyectar tendencias futuras se implementó un modelo ARIMA(p,d,q). La selección de parámetros óptimos se realizó mediante el criterio de información de Akaike (AICc). Las proyecciones cubren 2018–2022 para West Virginia.

## 3.7. Variables del Dataset

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv("NCHS_Leading_Causes.csv", dtype=str)
df.columns = ['year','cause_113','cause_name','state','deaths','age_adjusted_death_rate']
df['year'] = pd.to_numeric(df['year'], errors='coerce')
df['deaths'] = pd.to_numeric(df['deaths'].str.replace('.','',regex=False).str.replace(',','',regex=False), errors='coerce').fillna(0).astype(int)
df['age_adjusted_death_rate'] = pd.to_numeric(df['age_adjusted_death_rate'].str.replace(',','.'), errors='coerce')
df['state'] = df['state'].str.strip()
df['cause_name'] = df['cause_name'].str.strip()
df = df[(df['year']>=1999)&(df['year']<=2017)].dropna(subset=['year','age_adjusted_death_rate'])

estados = df[df['state']!='United States']
us = df[df['state']=='United States']
estados_2017 = estados[estados['year']==2017]
sin_all = estados[estados['cause_name']!='All causes']
print(f"Datos cargados: {len(df):,} filas | {df['state'].nunique()} entidades | {df['year'].min():.0f}–{df['year'].max():.0f}")


Datos cargados: 10,840 filas | 52 entidades | 1999–2017


In [2]:
print("Variables del dataset:")
for c in df.columns:
    print(f"  - {c}: {df[c].dtype}")
print(f"\nPeriodo: {int(df.year.min())} – {int(df.year.max())}")
print(f"Estados únicos: {df.state.nunique()}")
print(f"Causas únicas: {df.cause_name.nunique()}")
print(f"Total registros: {len(df):,}")

Variables del dataset:
  - year: int64
  - cause_113: str
  - cause_name: str
  - state: str
  - deaths: int64
  - age_adjusted_death_rate: float64

Periodo: 1999 – 2017
Estados únicos: 52
Causas únicas: 11
Total registros: 10,840
